## 9. CI/CD PIPELINE

Code are following AAI 540 Lab 6
## Setup the load from S3

In [1]:
# If you're using the default bucket, set DEFAULT_BUCKET = True; otherwise, if you're using a specific bucket, set it to False instead
DEFAULT_BUCKET = False

In [2]:
import boto3
import sagemaker
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TransformStep, TrainingStep #, ModelStep
from sagemaker.workflow.parameters import ParameterString
from sagemaker.workflow.model_step import ModelStep
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.transformer import Transformer
from sagemaker.model import Model
from sagemaker.workflow.conditions import ConditionLessThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.functions import Join
from sagemaker.model_metrics import MetricsSource, ModelMetrics
from sagemaker.inputs import TrainingInput, TransformInput, CreateModelInput
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.estimator import Estimator

import pandas as pd
import json
from pyathena import connect
sess = sagemaker.Session()
if DEFAULT_BUCKET is True:
    # Code to read/write using the default bucket
    bucket = sess.default_bucket()
else:
    # Code to use a previously existing bucket
    bucket = "usdmsaai540-spring2026-team1"
s3 = boto3.resource("s3")
role = sagemaker.get_execution_role()
region = boto3.Session().region_name
pipeline_session = PipelineSession()

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [3]:
# Set S3 path to Parquet data and S3 Prefix
s3_path_parquet = f's3://{bucket}/CCPP/data//parquet'
s3_prefix = 'CCPP-enery-prediction-linear-regression'

# Set Athena parameters
database_name = "ccpp_aws_fp"
table_name_csv = "final_data_csv"
table_name_parquet = "final_data_parquet"

# Preparing the input and batch data uris
#job_name = 'lr-2026-02-18-07-14-00' # Job name from V1
job_name = 'xg_2026-02-18-07-13-00' # Job name from V1

input_data_uri = f's3://{bucket}/{s3_prefix}/testing/' # Using the Testing data as input
batch_data_uri = f's3://{bucket}/{s3_prefix}/batch/' # Using the batch data as batch


In [4]:
# Preparing the model artifact parameters
model_artifact_param = ParameterString(
    name="ModelArtifactPath",
    default_value=f"s3://{bucket}/{s3_prefix}/{job_name}/{job_name}/output/model.tar.gz"
)

# Preparing the batch data parameters
batch_data_param = ParameterString(
    name="BatchDataPath",
    default_value=f"s3://{bucket}/{s3_prefix}/batch/"
)

from sagemaker.workflow.lambda_step import LambdaStep, LambdaOutput
from sagemaker.workflow.parameters import ParameterString
model = Model(
    image_uri=sagemaker.image_uris.retrieve(
        framework="linear-learner",
        region=region
    ),
    model_data=model_artifact_param,
    role=role
)


model_name_param = ParameterString(
    name="ModelName",
    default_value="existing-model-from-artifact"
)

step_register = LambdaStep(
    name="CreateModelViaLambda",
    lambda_func=lambda_handler,  # your deployed Lambda
    inputs={
        "ModelName": model_name_param,
        "ImageUri": model.image_uri,
        "ModelDataUrl": model_artifact_param,
        "RoleArn": role
    },
    outputs=[
        LambdaOutput(output_name="ModelName", output_type="String")
    ]
)

In [5]:
output_location = "s3://{}/{}/output/{}".format(bucket, s3_prefix, job_name)
output_location

's3://usdmsaai540-spring2026-team1/CCPP-enery-prediction-linear-regression/output/xg_2026-02-18-07-13-00'

## CI/CD Pipeline
### Preparing the model parameters

In [6]:
from sagemaker.workflow.parameters import (
    ParameterInteger,
    ParameterString,
    ParameterFloat,
)

processing_instance_count = ParameterInteger(name="ProcessingInstanceCount", default_value=1)
instance_type = ParameterString(name="TrainingInstanceType", default_value="ml.m5.xlarge")
model_approval_status = ParameterString(
    name="ModelApprovalStatus", default_value="PendingManualApproval"
)
input_data = ParameterString(
    name="InputData",
    default_value=input_data_uri,
)
batch_data = ParameterString(
    name="BatchData",
    default_value=batch_data_uri,
)
mse_threshold = ParameterFloat(name="MseThreshold", default_value=2.721507)
r2_threshold = ParameterFloat(name="R2Threshold", default_value=0.986821)

## Preparing a preprocessing step for the pipeline

In [7]:
!mkdir -p code

In [8]:
%%writefile code/preprocessing.py
import argparse
import os
import requests
import tempfile

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder


# Since we get a headerless CSV file, we specify the column names here.
feature_columns_names = [
    "gt_comp_dis_pressure",
    "gt_exhaust_pressure",
    "gt_inlet_temp",
    "gt_air_filter_diff_pressure",
    "solar_radiation",
    "solar_energy",
    "uv_index",
    "gt_ambient_pressure",
    "cloud_cover",
    "sea_level_pressure"
]
label_column = "gt_energy_yield"

feature_columns_dtype = {
    "gt_comp_dis_pressure": np.float64,
    "gt_exhaust_pressure": np.float64,
    "gt_inlet_temp": np.float64,
    "gt_air_filter_diff_pressure": np.float64,
    "solar_radiation": np.float64,
    "solar_energy": np.float64,
    "uv_index": np.float64,
    "gt_ambient_pressure": np.float64,
    "cloud_cover": np.float64,
    "sea_level_pressure": np.float64
}
label_column_dtype = {"gt_energy_yield": np.float64}


def merge_two_dicts(x, y):
    z = x.copy()
    z.update(y)
    return z


if __name__ == "__main__":
    base_dir = "/opt/ml/processing"

    df = pd.read_csv(
        f"{base_dir}/input/abalone-dataset.csv",
        header=None,
        names=feature_columns_names + [label_column],
        dtype=merge_two_dicts(feature_columns_dtype, label_column_dtype),
    )
    numeric_features = list(feature_columns_names)
    # Removing features with multi-colinearity
    numeric_features.remove("solar_energy","uv_index")
    numeric_transformer = Pipeline(
        steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
    )

    preprocess = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
        ]
    )

    y = df.pop("gt_energy_yield")
    X_pre = preprocess.fit_transform(df)
    y_pre = y.to_numpy().reshape(len(y), 1)

    X = np.concatenate((y_pre, X_pre), axis=1)

    np.random.shuffle(X)
    train, validation, test = np.split(X, [int(0.7 * len(X)), int(0.85 * len(X))])

    pd.DataFrame(train).to_csv(f"{base_dir}/train/train.csv", header=False, index=False)
    pd.DataFrame(validation).to_csv(
        f"{base_dir}/validation/validation.csv", header=False, index=False
    )
    pd.DataFrame(test).to_csv(f"{base_dir}/test/test.csv", header=False, index=False)

Overwriting code/preprocessing.py


In [9]:
# Configuring framework version
framework_version = "1.2-1"
# Defining a processor object
sklearn_processor = SKLearnProcessor(
    framework_version=framework_version,
    instance_type="ml.m5.xlarge",
    instance_count=processing_instance_count,
    base_job_name="sklearn-ccpp-process",
    role=role,
    sagemaker_session=pipeline_session,
)

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


In [10]:
# Preparing the arguments for the processor (by using the .run method, which is not running it directly - just obtaining the arguments)
processor_args = sklearn_processor.run(
    inputs=[
        ProcessingInput(source=input_data, destination="/opt/ml/processing/input"),
    ],
    outputs=[
        ProcessingOutput(output_name="train", source="/opt/ml/processing/train"),
        ProcessingOutput(output_name="validation", source="/opt/ml/processing/validation"),
        ProcessingOutput(output_name="test", source="/opt/ml/processing/test"),
    ],
    code="code/preprocessing.py",
)

step_process = ProcessingStep(name="CCPPProcess", step_args=processor_args)

/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


## Defining a Training step

In [11]:
# Preparing the configuration for the model path and training parameters
model_path = f"s3://{bucket}/CCPP/CCPPTrain"
image_uri = sagemaker.image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.0-1",
    py_version="py3",
    instance_type="ml.m5.xlarge",
)
xgb_train = Estimator(
    image_uri=image_uri,
    instance_type=instance_type,
    instance_count=1,
    output_path=model_path,
    role=role,
    sagemaker_session=pipeline_session,
)
xgb_train.set_hyperparameters(
    objective="reg:linear",
    num_round=50,
    max_depth=5,
    eta=0.2,
    gamma=4,
    min_child_weight=6,
    subsample=0.7,
)

train_args = xgb_train.fit(
    inputs={
        "train": TrainingInput(
            #s3_data=step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            s3_data=f's3://{bucket}/{s3_prefix}/train/',
            content_type="text/csv",
        ),
        "validation": TrainingInput(
            #s3_data=step_process.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
            s3_data=f's3://{bucket}/{s3_prefix}/validation/',
            content_type="text/csv",
        ),
    }
)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


In [12]:
# Preparing the training step
step_train = TrainingStep(
    name="CCPPTrain",
    step_args=train_args,
)

## Defining the evaluation script and step to evaluate the model

In [13]:
%%writefile code/evaluation.py
import json
import pathlib
import pickle
import tarfile

import joblib
import numpy as np
import pandas as pd
import xgboost

from sklearn.metrics import mean_squared_error, r2_score


if __name__ == "__main__":
    model_path = f"/opt/ml/processing/model/model.tar.gz"
    with tarfile.open(model_path) as tar:
        tar.extractall(path=".")

    #model = pickle.load(open("xgboost-model", "rb"))
    #model = pickle.load(open("model_algo-1", "rb"))
    model = xgboost.Booster()
    #model.load_model("model_algo-1")
    model.load_model("xgboost-model")
    

    test_path = "/opt/ml/processing/test/test_data.csv"
    df = pd.read_csv(test_path, header=None)

    y_test = df.iloc[:, -1].to_numpy()
    df.drop(df.columns[-1], axis=1, inplace=True)

    X_test = xgboost.DMatrix(df.values)

    predictions = model.predict(X_test)

    mse = mean_squared_error(y_test, predictions)
    std = np.std(y_test - predictions)
    
    r2 = r2_score(y_test, predictions)
    
    report_dict = {
        "regression_metrics": {
            "mse": {
                "value": mse,
                "standard_deviation": std
            },
            "r2": {
                "value": r2
            },
        },
    }
    output_dir = "/opt/ml/processing/evaluation"
    pathlib.Path(output_dir).mkdir(parents=True, exist_ok=True)

    evaluation_path = f"{output_dir}/evaluation.json"
    with open(evaluation_path, "w") as f:
        f.write(json.dumps(report_dict))

Overwriting code/evaluation.py


In [14]:
# Configuring the Evalution script and evalution arguments
script_eval = ScriptProcessor(
    image_uri=image_uri,
    command=["python3"],
    instance_type="ml.m5.xlarge",
    instance_count=1,
    base_job_name="script-ccpp-eval",
    role=role,
    sagemaker_session=pipeline_session,
)

eval_args = script_eval.run(
    inputs=[
        ProcessingInput(
            #source=step_train.properties.ModelArtifacts.S3ModelArtifacts, ##This artifact may be naming the model differently!!!
            #source= FIGURE OUT HOW TO PASS THE MODEL STORED IN S3 HERE!! Alternatively, Professor suggested to copy the model locally in the same folder
            source=f's3://{bucket}/{s3_prefix}/output/{job_name}/{job_name}/output/model.tar.gz',
            destination="/opt/ml/processing/model",
        ),
        ProcessingInput(
            #source=step_process.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            source=f's3://{bucket}/{s3_prefix}/testing/',
            destination="/opt/ml/processing/test",
        ),
    ],
    outputs=[
        ProcessingOutput(output_name="evaluation", source="/opt/ml/processing/evaluation"),
    ],
    code="code/evaluation.py",
)

In [15]:
# Configuring an evaluation report to store the results of the evaluation step
evaluation_report = PropertyFile(
    name="EvaluationReport", output_name="evaluation", path="evaluation.json"
)

# Configuring the evaluation step
step_eval = ProcessingStep(
    name="CCPPEval",
    step_args=eval_args,
    property_files=[evaluation_report],
)

In [16]:
# Defining the model from the existing model.tar.gz file
model = Model(
    image_uri=sagemaker.image_uris.retrieve(
        framework="linear-learner",
        region=region
    ),
    #model_data=model_artifact_param,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    sagemaker_session=pipeline_session,
    role=role
)

INFO:sagemaker.image_uris:Same images used for training and inference. Defaulting to image scope: inference.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


In [17]:
# Configuring the create model step
step_create_model = ModelStep(
    name="CCPPCreateModel",
    step_args=model.create(instance_type="ml.m5.large", accelerator_type="ml.eia1.medium"),
)

In [18]:
# Configuring the Transformer for inference
transformer = Transformer(
    model_name=step_create_model.properties.ModelName,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    output_path=f"s3://{bucket}/{s3_prefix}/CCPPTransform",
)

In [19]:
# Defining the Transform Step
step_transform = TransformStep(
    name="CCPPTransform", transformer=transformer, inputs=TransformInput(data=batch_data)
)

In [20]:
model_package_group_name = f"CCPPModelPackageGroupName"
# Defining the metrics to evaluate
model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri="{}/evaluation.json".format(
            step_eval.arguments["ProcessingOutputConfig"]["Outputs"][0]["S3Output"]["S3Uri"]
        ),
        content_type="application/json",
    )
)

register_args = model.register(
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.t2.medium", "ml.m5.xlarge"],
    transform_instances=["ml.m5.xlarge"],
    model_package_group_name=model_package_group_name,
    approval_status=model_approval_status,
    model_metrics=model_metrics,
)
step_register = ModelStep(name="CCPPRegisterModel", step_args=register_args)

In [21]:
# Defining the Fail Step
#step_fail = FailStep(
#    name="CCPPMSEAndR2Fail",
#    error_message=Join(
#        on=" ",
#        values=[
#            "Execution failed due to:",
#            "MSE <", mse_threshold,
#            "or",
#            "R2 <", r2_threshold
#        ]
#    ),
#)
step_fail = FailStep(
    name="CCPPMSEAndR2Fail",
    error_message=Join(
        on=" ",
        values=[
            "Execution failed due to:",
            # MSE actual value
            "MSE value:",
            JsonGet(
                step_name=step_eval.name,
                property_file=evaluation_report,
                json_path="regression_metrics.mse.value",
            ),
            "exceeded threshold:",
            mse_threshold,

            "and/or",

            # R2 actual value
            "R2 value:",
            JsonGet(
                step_name=step_eval.name,
                property_file=evaluation_report,
                json_path="regression_metrics.r2.value",
            ),
            "fell below threshold:",
            r2_threshold,
        ]
    ),
)

## Defining a condition step to check MSE and R-Squared of the model to decide the pipeline's course of action

In [22]:
# Creating a "less than or equal" condition for the mse
cond_lte_mse = ConditionLessThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="regression_metrics.mse.value",
    ),
    right=mse_threshold,
)

# Creating a "less than or equal" condition for the R-squared
cond_lte_r2 = ConditionLessThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="regression_metrics.r2.value",
    ),
    right=r2_threshold,
)
step_cond = ConditionStep(
    name="CCPPMSEAndR2Cond",
    conditions=[cond_lte_mse, cond_lte_r2],
    if_steps=[step_register, step_create_model, step_transform],
    else_steps=[step_fail],
)

## Defining a pipeline using the parameters, steps, and conditions defined above

In [23]:
# Defining the pipeline with all the parameters defined above
pipeline_name = f"CCPPPipeline"
pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        processing_instance_count,
        instance_type,
        model_approval_status,
        input_data,
        batch_data,
        mse_threshold,
        r2_threshold,
    ],
    #steps=[step_process, step_train, step_eval, step_cond],
    steps=[step_train, step_eval, step_cond],
)

In [24]:
# Confirming the pipeline is well-defined and the parameters and step properties resolved correctly
definition = json.loads(pipeline.definition())
definition

{'Version': '2020-12-01',
 'Metadata': {},
 'Parameters': [{'Name': 'ProcessingInstanceCount',
   'Type': 'Integer',
   'DefaultValue': 1},
  {'Name': 'TrainingInstanceType',
   'Type': 'String',
   'DefaultValue': 'ml.m5.xlarge'},
  {'Name': 'ModelApprovalStatus',
   'Type': 'String',
   'DefaultValue': 'PendingManualApproval'},
  {'Name': 'InputData',
   'Type': 'String',
   'DefaultValue': 's3://usdmsaai540-spring2026-team1/CCPP-enery-prediction-linear-regression/testing/'},
  {'Name': 'BatchData',
   'Type': 'String',
   'DefaultValue': 's3://usdmsaai540-spring2026-team1/CCPP-enery-prediction-linear-regression/batch/'},
  {'Name': 'MseThreshold', 'Type': 'Float', 'DefaultValue': 2.721507},
  {'Name': 'R2Threshold', 'Type': 'Float', 'DefaultValue': 0.986821}],
 'PipelineExperimentConfig': {'ExperimentName': {'Get': 'Execution.PipelineName'},
  'TrialName': {'Get': 'Execution.PipelineExecutionId'}},
 'Steps': [{'Name': 'CCPPTrain',
   'Type': 'Training',
   'Arguments': {'AlgorithmSp

## Submitting the pipeline to SageMaker and starting execution

In [25]:
# Submitting the pipeline definition to the Pipeline service, using the session role to create all the jobs defined in the steps
pipeline.upsert(role_arn=role)

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:987390971271:pipeline/CCPPPipeline',
 'ResponseMetadata': {'RequestId': '5baefafb-ebf2-4706-b2bb-31f9cb99b926',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '5baefafb-ebf2-4706-b2bb-31f9cb99b926',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '103',
   'date': 'Fri, 20 Feb 2026 07:30:31 GMT'},
  'RetryAttempts': 0}}

In [26]:
# Starting the pipeline and accepting all the default parameters
execution = pipeline.start()

In [27]:
# Describing the pipeline execution
execution.describe()

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:987390971271:pipeline/CCPPPipeline',
 'PipelineExecutionArn': 'arn:aws:sagemaker:us-east-1:987390971271:pipeline/CCPPPipeline/execution/jxnsx9wk2tzb',
 'PipelineExecutionDisplayName': 'execution-1771572631357',
 'PipelineExecutionStatus': 'Executing',
 'CreationTime': datetime.datetime(2026, 2, 20, 7, 30, 31, 276000, tzinfo=tzlocal()),
 'LastModifiedTime': datetime.datetime(2026, 2, 20, 7, 30, 31, 276000, tzinfo=tzlocal()),
 'CreatedBy': {'UserProfileArn': 'arn:aws:sagemaker:us-east-1:987390971271:user-profile/d-us8baws1boci/default-1770785083214',
  'UserProfileName': 'default-1770785083214',
  'DomainId': 'd-us8baws1boci',
  'IamIdentity': {'Arn': 'arn:aws:sts::987390971271:assumed-role/LabRole/SageMaker',
   'PrincipalId': 'AROA6LZIWRWD4A4MFA6UD:SageMaker'}},
 'LastModifiedBy': {'UserProfileArn': 'arn:aws:sagemaker:us-east-1:987390971271:user-profile/d-us8baws1boci/default-1770785083214',
  'UserProfileName': 'default-1770785083214',
  'D

In [30]:
# Waiting for the execution to complete
try:
    execution.wait()
except Exception as error:
    print(error)

Waiter PipelineExecutionComplete failed: Waiter encountered a terminal failure state: For expression "PipelineExecutionStatus" we matched expected path: "Failed"


In [29]:
# Listing the steps in the execution or confirmation
execution.list_steps()

[{'StepName': 'CCPPMSEAndR2Fail',
  'StartTime': datetime.datetime(2026, 2, 20, 7, 33, 6, 945000, tzinfo=tzlocal()),
  'EndTime': datetime.datetime(2026, 2, 20, 7, 33, 7, 228000, tzinfo=tzlocal()),
  'StepStatus': 'Failed',
  'FailureReason': 'Execution failed due to: MSE value: 3.90598671066806 exceeded threshold: 2.721507 and/or R2 value: 0.8994239605182245 fell below threshold: 0.986821',
  'Metadata': {'Fail': {'ErrorMessage': 'Execution failed due to: MSE value: 3.90598671066806 exceeded threshold: 2.721507 and/or R2 value: 0.8994239605182245 fell below threshold: 0.986821'}},
  'AttemptCount': 1},
 {'StepName': 'CCPPMSEAndR2Cond',
  'StartTime': datetime.datetime(2026, 2, 20, 7, 33, 6, 472000, tzinfo=tzlocal()),
  'EndTime': datetime.datetime(2026, 2, 20, 7, 33, 6, 702000, tzinfo=tzlocal()),
  'StepStatus': 'Succeeded',
  'Metadata': {'Condition': {'Outcome': 'False'}},
  'AttemptCount': 1},
 {'StepName': 'CCPPTrain',
  'StartTime': datetime.datetime(2026, 2, 20, 7, 30, 32, 29300

## Kernel Shutdown

%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>